# 00 · Setup: exerciții

**Teorie:** [docs/00_setup/READING.md](../../docs/00_setup/READING.md) și [SETUP.md §6: checkpoint-uri](../../docs/00_setup/SETUP.md)
**Rulează pe:** Mac (CPU/MPS), gratuit
**Ce exersezi:** login pe HF, repo privat, salvarea checkpoint-urilor pe Hub, reluarea unui training întrerupt.

**Cerințe:**
- ai făcut `hf auth login` cu un token cu drept de **write**;
- ai instalat `transformers`, `datasets`, `accelerate` și `huggingface_hub`.

**Reguli:**
- scrii codul în celulele `# TODO`;
- rulezi celula **✅ Verificare** de sub fiecare exercițiu;
- deschizi indiciul doar dacă te blochezi.

## Exercițiul 1: Cine ești pe Hub?

Folosește `whoami()` din `huggingface_hub` și salvează numele tău de utilizator în `username`.

In [ ]:
# TODO: importă whoami și salvează numele de utilizator în `username`
username = None

<details><summary>💡 Indiciu</summary>

`whoami()` returnează un dicționar. Numele e la cheia `"name"`.

</details>

In [ ]:
# ✅ Verificare
assert isinstance(username, str) and username, "username trebuie să fie numele tău de pe HF"
print(f"✅ Logat ca {username}")

## Exercițiul 2: Un repo privat de test

Creează repo-ul privat `<username>/practice-checkpoints`. Folosește `exist_ok=True`, ca celula să poată fi rulată de mai multe ori. Salvează id-ul în `repo_id`.

In [ ]:
# TODO: creează repo-ul privat și salvează id-ul în `repo_id`
repo_id = None

<details><summary>💡 Indiciu</summary>

`from huggingface_hub import create_repo`. Argumentele care contează: `private=True`, `exist_ok=True`.

</details>

In [ ]:
# ✅ Verificare
from huggingface_hub import HfApi
info = HfApi().repo_info(repo_id)
assert info.private, "Repo-ul trebuie să fie privat"
print(f"✅ Repo privat: https://huggingface.co/{repo_id}")

## Exercițiul 3: Date și un model minuscul

Folosim `sshleifer/tiny-gpt2`, un model care nu învață nimic util, dar e atât de mic încât checkpoint-urile urcă instant. Aici exersezi **mecanismul**, nu calitatea modelului.

In [ ]:
# Date de pornire (nu modifica)
texts = [
    "the cat sat on the mat",
    "speech recognition turns audio into text",
    "whisper uses log mel spectrograms",
    "qwen is a large language model",
    "checkpoints let you resume training",
] * 8   # 40 de propoziții
MODEL_ID = "sshleifer/tiny-gpt2"

In [ ]:
# TODO 1: încarcă tokenizer-ul pentru MODEL_ID și setează tokenizer.pad_token = tokenizer.eos_token
# TODO 2: creează un Dataset din `texts`, tokenizează-l și păstrează doar coloanele tokenizate -> `train_ds`
# TODO 3: creează `collator`, un data collator pentru language modeling cauzal (fără masked LM)
tokenizer = None
train_ds = None
collator = None

<details><summary>💡 Indiciu</summary>

- `Dataset.from_dict({"text": texts})`, apoi `.map(..., batched=True, remove_columns=["text"])`
- `DataCollatorForLanguageModeling(tokenizer, mlm=False)`

</details>

In [ ]:
# ✅ Verificare
assert len(train_ds) == 40, "Ar trebui să ai 40 de exemple"
assert "input_ids" in train_ds.column_names and "text" not in train_ds.column_names
assert collator.mlm is False, "Pentru GPT (cauzal) mlm trebuie să fie False"
print("✅ Date pregătite:", train_ds.column_names)

## Exercițiul 4: Antrenare cu checkpoint-uri pe Hub

Scrie o funcție `make_trainer(max_steps)` care încarcă **de fiecare dată un model nou** și returnează un `Trainer` cu:

| parametru | valoare |
|---|---|
| `output_dir` | `"outputs/run"` |
| `max_steps` | argumentul funcției |
| `per_device_train_batch_size` | `4` |
| `save_strategy` / `save_steps` / `save_total_limit` | `"steps"` / `5` / `2` |
| `logging_steps` | `1` |
| `report_to` | `"none"` |
| `push_to_hub` / `hub_model_id` | `True` / `repo_id` |
| `hub_strategy` | `"checkpoint"`: ultimul checkpoint ajunge și în `last-checkpoint/` |
| `hub_private_repo` | `True` |

Apoi rulează `trainer = make_trainer(10)` și `trainer.train()`.

Funcția o refolosim la exercițiul 5, ca să simulăm o sesiune nouă.

In [ ]:
# TODO: scrie make_trainer(max_steps), apoi antrenează 10 pași
def make_trainer(max_steps):
    ...

trainer = None

<details><summary>💡 Indiciu</summary>

`AutoModelForCausalLM.from_pretrained(MODEL_ID)` în interiorul funcției. Apoi `TrainingArguments(...)` și `Trainer(model=..., args=..., train_dataset=train_ds, data_collator=collator)`.

</details>

In [ ]:
# ✅ Verificare (push-ul e asincron: dacă pică, așteaptă câteva secunde și rulează din nou)
from huggingface_hub import list_repo_files
ck = sorted(f for f in list_repo_files(repo_id) if f.startswith("last-checkpoint/"))
assert ck, "Încă nu e nimic în last-checkpoint/ pe Hub. Așteaptă câteva secunde și rulează din nou celula."
print("✅ Pe Hub, în last-checkpoint/:", [f.split("/", 1)[1] for f in ck])

## Exercițiul 5: Sesiune nouă, reluare de pe Hub

Imaginează-ți că serverul Colab a fost șters și n-a rămas nimic local. Pașii:
1. Șterge folderul local `outputs/` (`shutil.rmtree`). De acum, singura copie e cea de pe Hub.
2. Descarcă **doar** `last-checkpoint/` cu `snapshot_download` în `outputs/from_hub`. Salvează calea către checkpoint în `ckpt_dir`.
3. Creează `trainer2 = make_trainer(20)` și reia cu `trainer2.train(resume_from_checkpoint=ckpt_dir)`.

Urmărește în log de la ce pas pornește.

In [ ]:
# TODO: șterge outputs/, descarcă last-checkpoint/, reia până la pasul 20
ckpt_dir = None
trainer2 = None

<details><summary>💡 Indiciu</summary>

`snapshot_download(repo_id, allow_patterns=["last-checkpoint/*"], local_dir="outputs/from_hub")`. Checkpoint-ul se află apoi în `outputs/from_hub/last-checkpoint`.

</details>

In [ ]:
# ✅ Verificare
import json, os
state = json.load(open(os.path.join(ckpt_dir, "trainer_state.json")))
assert state["global_step"] == 10, f"Checkpoint-ul descărcat e de la pasul {state['global_step']}, nu 10. Așteaptă puțin și descarcă din nou."
assert trainer2.state.global_step == 20, "trainer2 ar trebui să termine la pasul 20"
print("✅ Reluat de la pasul 10 și terminat la pasul 20")

## Exercițiul 6: Ce conține un checkpoint?

Listează fișierele din `ckpt_dir` în `ckpt_files`, apoi scrie în celula de mai jos, cu cuvintele tale, rolul fiecărui fișier.

In [ ]:
# TODO: lista de fișiere din ckpt_dir -> ckpt_files
ckpt_files = None

In [ ]:
# ✅ Verificare
expected = {"model.safetensors", "optimizer.pt", "scheduler.pt", "trainer_state.json"}
assert expected <= set(ckpt_files), f"Lipsesc: {expected - set(ckpt_files)}"
print("✅", sorted(ckpt_files))

✍️ **Răspunsul tău** (o propoziție pe fișier):
- `model.safetensors`:
- `optimizer.pt`:
- `scheduler.pt`:
- `trainer_state.json`:
- `rng_state.pth`:

De ce n-ar fi fost suficient doar `model.safetensors` ca să reiei corect?

## Curățenie (opțional)

Dacă nu mai ai nevoie de repo-ul de test: `from huggingface_hub import delete_repo; delete_repo(repo_id)`.